# 02 — Same geometry, different optimization

Analysis 6.2 and main Figures 2–3. All arms use the same fixed PCA subspace; only the orthogonal gauge policy changes. Figure 2 reports the downstream distribution across Haar rotations against deterministic fitting policies; Figure 3 isolates the instantaneous pre/post-refit change on the frozen gauge subset. Gram error and linear CKA remain numeric negative controls rather than a separate heatmap.


In [ ]:
# 1. Cấu hình
from pathlib import Path
REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"
AUTO_PULL_REPO = True
INSTALL_REQUIREMENTS = True
PAIR = "qwen3_4b_to_bert_base"
TRAIN_DATA_REL = Path("data/train_set/train_100k.csv")
RUN_NAME = f"analysis_rotation_{PAIR}_v1"
SEEDS = [42, 43, 44]
ROTATION_DRAWS = list(range(10))
BATCH_SIZE, EPOCHS, LR = 128, 5, 7e-5
TARGET_ROWS = 256
EXECUTE = False
CUDA_VISIBLE_DEVICES = "0"


In [ ]:
# 2. Clone/fetch repo, dependencies, imports và output
import shlex, subprocess, sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
cwd = Path.cwd().resolve()
PROJECT_DIR = next((p for p in (cwd, cwd.parent) if (p / "main.py").is_file()), None)
if PROJECT_DIR is None:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    assert (PROJECT_DIR / "main.py").is_file(), f"Repo không hợp lệ: {PROJECT_DIR}"
if AUTO_PULL_REPO:
    dirty = subprocess.run(["git", "-C", str(PROJECT_DIR), "status", "--porcelain", "--untracked-files=no"], check=True, capture_output=True, text=True).stdout.strip()
    if dirty:
        print("[git] Bỏ qua pull vì repo có tracked changes.")
    else:
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
if INSTALL_REQUIREMENTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
git_head = subprocess.run(["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
sys.path[:0] = [str(PROJECT_DIR), str(PROJECT_DIR / "notebooks")]
from _analysis_common import PAIRS, collect_jobs, geoode_command, load_teacher_cache, read_jsonl, run_jobs, set_paper_style, teacher_cache_path
from src import structural_audit as audit
PAIR_CONFIG = PAIRS[PAIR]
TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
assert TRAIN_DATA.is_file(), f"Thiếu training data: {TRAIN_DATA}"
RUN_ROOT = PROJECT_DIR / "runs" / RUN_NAME
CACHE_DIR = PROJECT_DIR / "runs" / "teacher_cache"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
set_paper_style()
print(f"Repo: {PROJECT_DIR} @ {git_head}")
print(f"Output: {RUN_ROOT}")


In [ ]:
# 3. PCA gauge, Haar gauges, one-off Procrustes và epoch-wise refit
specs = [
    {"arm": "pca_proc", "draws": [None], "refit": 0, "args": ["--gauge_align", "--gauge_rotation", "procrustes"]},
    {"arm": "pca_proc_refit", "draws": [None], "refit": 1, "args": ["--gauge_align", "--gauge_rotation", "procrustes"]},
    {"arm": "pca_none", "draws": [None], "refit": 0, "args": ["--no-gauge_align"]},
    {"arm": "pca_random", "draws": ROTATION_DRAWS, "refit": 0, "args": ["--gauge_align", "--gauge_rotation", "random"]},
]
jobs = []
for spec in specs:
    for draw in spec["draws"]:
        for seed in SEEDS:
            cell = spec["arm"] if draw is None else f"{spec['arm']}__d{draw}"
            run_dir = RUN_ROOT / cell / f"seed_{seed}"
            extra = ["--projection_type", "pca", "--lambda_end", "1", "--lambda_ctr", "0", "--lambda_topo", "0", "--gauge_refit_every", str(spec["refit"]), "--diag_every", "50", "--no_eval_retrieval", *spec["args"]]
            if draw is not None:
                extra += ["--gauge_random_seed", str(draw)]
            jobs.append({"name": f"{cell}/seed_{seed}", "arm": spec["arm"], "draw": draw, "seed": seed, "run_dir": run_dir, "command": geoode_command(PROJECT_DIR, pair=PAIR_CONFIG, train_data=TRAIN_DATA, cache_dir=CACHE_DIR, run_dir=run_dir, seed=seed, batch_size=BATCH_SIZE, epochs=EPOCHS, learning_rate=LR, extra=extra)})
print(f"Plan: {len(jobs)} jobs")
for job in jobs:
    print(shlex.join(job["command"]))
if EXECUTE:
    display(run_jobs(PROJECT_DIR, jobs, cuda_visible_devices=CUDA_VISIBLE_DEVICES))


In [ ]:
# 4. Verify invariance on the maps actually saved by the runs
results = collect_jobs(jobs)
results.to_csv(RUN_ROOT / "rotation_results.csv", index=False)
done = results.query("status == 'done'").copy()
display(done[[c for c in ["arm", "draw", "seed", "cos_after", "avg_all"] if c in done]])
cache = teacher_cache_path(PROJECT_DIR, CACHE_DIR, pair=PAIR_CONFIG, train_data=TRAIN_DATA)
teacher, _ = load_teacher_cache(cache)
teacher = teacher[:min(TARGET_ROWS, len(teacher))].float()
def saved_targets(arm, draw=None):
    cell = arm if draw is None else f"{arm}__d{draw}"
    path = RUN_ROOT / cell / f"seed_{SEEDS[0]}" / "teacher_projection.pt"
    if not path.is_file():
        return None
    return audit.targets_from_saved(teacher, torch.load(path, map_location="cpu", weights_only=False))
reference = saved_targets("pca_none")
variants = {"No rotation": reference, "One-off Procrustes": saved_targets("pca_proc"), "Epoch-refit Procrustes": saved_targets("pca_proc_refit"), "Haar Q": saved_targets("pca_random", ROTATION_DRAWS[0])}
invariance = []
if reference is not None:
    for name, target in variants.items():
        if target is not None:
            invariance.append({"variant": name, "gram_rmse": audit.gram_rmse(reference, target), "linear_cka": audit.linear_cka(reference, target)})
invariance = pd.DataFrame(invariance)
invariance.to_csv(RUN_ROOT / "rotation_invariance.csv", index=False)
display(invariance.style.format({"gram_rmse": "{:.3e}", "linear_cka": "{:.6f}"}))


In [ ]:
# 5a. Figure 2 — same geometry, different KD outcomes
labels = {"pca_none": "PCA gauge", "pca_random": "Haar rotations", "pca_proc": "Fit once", "pca_proc_refit": "Refit / epoch"}
order = ["pca_none", "pca_random", "pca_proc", "pca_proc_refit"]
if done.empty:
    print("No completed rotation runs yet.")
else:
    fig, ax = plt.subplots(figsize=(5.5, 2.45))
    random_scores = 100 * done.query("arm == 'pca_random'")["avg_all"].dropna().to_numpy()
    if len(random_scores):
        box = ax.boxplot([random_scores], positions=[1], widths=.48, patch_artist=True, showfliers=False)
        box["boxes"][0].set(facecolor="#D4D8DE", edgecolor="#6B7280")
        for item in box["medians"] + box["whiskers"] + box["caps"]: item.set(color="#1F2937", linewidth=1.2)
    colors = {"pca_none": "#6B7280", "pca_proc": "#2B6CB0", "pca_proc_refit": "#DD6B20"}
    for position, arm in enumerate(order):
        if arm == "pca_random": continue
        values = 100 * done.query("arm == @arm")["avg_all"].dropna()
        if len(values): ax.errorbar(position, values.mean(), yerr=values.std(ddof=1), fmt="o", ms=6, capsize=3, color=colors[arm], zorder=3)
    ax.set(xticks=range(len(order)), xticklabels=[labels[arm] for arm in order], ylabel="Final average score")
    ax.grid(axis="y", color="#E5E7EB", lw=.7); ax.spines[["top", "right"]].set_visible(False)
    if not invariance.empty:
        ax.text(.02, .97, f"orthogonal gauges: CKA ≥ {invariance.linear_cka.min():.6f}\nmax Gram RMSE = {invariance.gram_rmse.max():.1e}", transform=ax.transAxes, va="top", fontsize=7, color="#4B5563")
    fig.tight_layout()
    fig.savefig(RUN_ROOT / "figure_2_same_geometry_different_kd.pdf", bbox_inches="tight")
    fig.savefig(RUN_ROOT / "figure_2_same_geometry_different_kd.png", dpi=300, bbox_inches="tight")
    plt.show()

# 5b. Figure 3 — exact refit events on the frozen calibration subset
event_rows = []
for job in jobs:
    if job["arm"] != "pca_proc_refit": continue
    path = Path(job["run_dir"]) / "teacher_projection.pt"
    if not path.is_file(): continue
    history = torch.load(path, map_location="cpu", weights_only=False).get("gauge_history", [])
    for event in history:
        if "epoch" not in event or "cos_previous_gauge" not in event: continue
        event_rows += [
            {"seed": job["seed"], "epoch": event["epoch"], "state": "Before refit", "alignment_error": 1 - event["cos_previous_gauge"]},
            {"seed": job["seed"], "epoch": event["epoch"], "state": "After refit", "alignment_error": 1 - event["cos_after"]},
        ]
events = pd.DataFrame(event_rows)
events.to_csv(RUN_ROOT / "gauge_refit_events.csv", index=False)
if not events.empty:
    fig, ax = plt.subplots(figsize=(5.5, 2.45))
    offsets = {"Before refit": -.055, "After refit": .055}
    colors = {"Before refit": "#6B7280", "After refit": "#DD6B20"}
    stats = events.groupby(["state", "epoch"])["alignment_error"].agg(["mean", "std"]).reset_index()
    for state in ["Before refit", "After refit"]:
        part = stats.query("state == @state").sort_values("epoch")
        ax.errorbar(part["epoch"] + offsets[state], part["mean"], yerr=part["std"].fillna(0), marker="o", capsize=2.5, color=colors[state], label=state)
    for epoch in sorted(events["epoch"].unique()):
        pair = stats.query("epoch == @epoch").set_index("state")["mean"]
        if set(offsets).issubset(pair.index): ax.plot([epoch + offsets["Before refit"], epoch + offsets["After refit"]], [pair["Before refit"], pair["After refit"]], color="#C7CBD1", lw=1, zorder=0)
    ax.set(xlabel="Epoch boundary", ylabel="Frozen-subset alignment error ↓", xticks=sorted(events["epoch"].unique()))
    ax.grid(axis="y", color="#E5E7EB", lw=.7); ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False, ncol=2)
    fig.tight_layout()
    fig.savefig(RUN_ROOT / "figure_3_refit_restores_interface.pdf", bbox_inches="tight")
    fig.savefig(RUN_ROOT / "figure_3_refit_restores_interface.png", dpi=300, bbox_inches="tight")
    plt.show()
